# GOLD ATP RANKING

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [ ]:
try:
    spark = SparkSession.builder.appName("fact_player_ranking").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
# silver
tb_player_match = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

# gold
tb_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Ranking

In [ ]:
df_rank_events = (
    tb_player_match.alias("m")
    .join(
        tb_tournaments.alias("t"),
        f.col("m.TOURNEY_ID") == f.col("t.TOURNEY_ID"),
        "inner"
    )
    .select(
        f.col("m.PLAYER_ID").alias("EVENT_PLAYER_ID"),
        f.to_date(f.col("t.TOURNEY_START_DATE").cast("string")).alias("EVENT_DATE"),
        f.col("m.PLAYER_RANK_PTS").cast("int").alias("EVENT_PTS")
    )
    .filter(
        f.col("EVENT_PLAYER_ID").isNotNull() & 
        f.col("EVENT_PTS").isNotNull() & 
        f.col("EVENT_DATE").isNotNull()
    )
    .groupBy("EVENT_PLAYER_ID", "EVENT_DATE")
    .agg(f.max("EVENT_PTS").alias("EVENT_PTS"))
)

df_player_lifecycle = (
    df_rank_events
    .groupBy("EVENT_PLAYER_ID")
    .agg(
        f.min("EVENT_DATE").alias("FIRST_MATCH_DATE"),
        f.max("EVENT_DATE").alias("LAST_MATCH_DATE")
    )
)

df_calendar_weeks = (
    tb_tournaments
    .select(
        f.to_date(f.col("TOURNEY_START_DATE").cast("string")).alias("RANKING_DATE")
    )
    .filter(f.col("RANKING_DATE").isNotNull())
    .distinct()
)

df_grid = (
    df_player_lifecycle
    .crossJoin(df_calendar_weeks)
    .filter(
        (f.col("RANKING_DATE") >= f.col("FIRST_MATCH_DATE")) &
        (f.col("RANKING_DATE") <= f.date_add(f.col("LAST_MATCH_DATE"), 365))
    )
    .select(
        f.col("EVENT_PLAYER_ID").alias("PLAYER_ID"),
        f.col("RANKING_DATE")
    )
)

window_ffill = (
    Window.partitionBy("PLAYER_ID")
    .orderBy("RANKING_DATE")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_filled = (
    df_grid
    .join(
        df_rank_events,
        (df_grid.PLAYER_ID == df_rank_events.EVENT_PLAYER_ID) & 
        (df_grid.RANKING_DATE == df_rank_events.EVENT_DATE),
        "left"
    )
    .select(
        f.col("RANKING_DATE"),
        f.col("PLAYER_ID"),
        f.last("EVENT_DATE", ignorenulls=True).over(window_ffill).alias("LAST_MATCH_DATE"),
        f.last("EVENT_PTS", ignorenulls=True).over(window_ffill).alias("LAST_KNOWN_PTS")
    )
    # Expira pontos se o atleta ficou mais de 52 semanas sem jogar
    .withColumn("DAYS_INACTIVE", f.datediff(f.col("RANKING_DATE"), f.col("LAST_MATCH_DATE")))
    .withColumn(
        "ACTIVE_PTS",
        f.when(f.col("DAYS_INACTIVE") > 365, f.lit(0)).otherwise(f.col("LAST_KNOWN_PTS"))
    )
    .filter(f.col("ACTIVE_PTS") > 0)
)

window_weekly_rank = Window.partitionBy("RANKING_DATE").orderBy(f.col("ACTIVE_PTS").desc())

df = (
    df_filled.alias("f")
    .join(
            tb_players.alias("p"),
            (f.col("f.PLAYER_ID").cast("string") == f.col("p.PLAYER_ID").cast("string"))
            | (
                f.col("f.PLAYER_ID").cast("string")
                == f.col("p.PLAYER_ID_OLD").cast("string")
            ),
            "left",
    )
    .withColumn("PLAYER_RANK", f.dense_rank().over(window_weekly_rank))
    .select(
        f.col("p.SK_PLAYER"),
        f.col("PLAYER_RANK"),
        f.col("ACTIVE_PTS").alias("PLAYER_RANK_PTS"),
        f.col("RANKING_DATE")
    )
)

## Save dataframe

### Local

In [19]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_ranking.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)